# 🚀 POLYDIM-EINSOF: Benchmark Científico en Silicio (1-Clic)
### Autor: Ariel García Traba (Independent Research / POLYDIM-EINSOF Initiative)
### Repositorio Oficial: [https://github.com/AGT1973/POLYDIM_EINSOF](https://github.com/AGT1973/POLYDIM_EINSOF)
### Hugging Face Hub: [https://huggingface.co/polydim-cla/polydim-stiefel-manifold-adapters](https://huggingface.co/polydim-cla/polydim-stiefel-manifold-adapters)

Este cuaderno compara empíricamente en tu máquina/GPU el rendimiento de:
1. **Modelo Común (Baseline 1D):** Comunicación tradicional mediante texto y tokenizador (cuello de botella de Shannon DPI).
2. **Modelo con POLYDIM (PMTP):** Comunicación continua directa en la **Variedad de Stiefel** $St(d_{in}, d_{out}) \subset S^{D-1}$.

---

> ⚡ **PASO PREVIO OBLIGATORIO (Para activar la GPU T4 gratuita o TPU):**
> 1. En el menú superior de Google Colab, andá a: `Entorno de ejecución` ➔ `Cambiar tipo de entorno de ejecución`.
> 2. En el menú desplegable **Acelerador de hardware**, seleccioná **GPU** (la placa T4 es 100% gratuita) o **TPU**.
> 3. Hacé clic en **Guardar**.
> 4. ¡Listo! Ahora podés ejecutar las celdas abajo para medir la velocidad sobre silicio real acelerado.

In [ ]:
# [1] Instalación de dependencias (solo tarda unos segundos)
!pip install -q torch transformers safetensors huggingface_hub accelerate

In [ ]:
# [2] Ejecución del Benchmark Comparativo sobre Silicio Detectado
import time
import torch
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer

print("=" * 70)
print("INICIANDO EVALUACIÓN COMPARATIVA: MODELO COMÚN vs MODELO CON POLYDIM")
print("=" * 70)

# 0. Detección automática del silicio activo (GPU CUDA vs CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU del Host"
print(f"\n[SILICIO ACTIVO DETECTADO]: {device.upper()} ({device_name})")
if device == "cpu":
    print("  * TIP: Podés activar GPU gratuita en: Entorno de ejecución -> Cambiar tipo de entorno -> GPU")

# 1. Descarga del Adaptador Stiefel oficial de Hugging Face
print("\n1. Descargando Adaptador de Stiefel (polydim-cla)... ")
adapter_file = hf_hub_download(
    repo_id="polydim-cla/polydim-stiefel-manifold-adapters",
    filename="2_qwen2_5_PMTP_v1/adapter_weights.safetensors"
)
weights = load_file(adapter_file)
W_stiefel = weights["W"].float().to(device) # Shape: (1024, 896)
print(f"   [OK] Adaptador Stiefel cargado: matriz {W_stiefel.shape} en {device.upper()}")

# 2. Carga del Modelo Base (Qwen 0.5B con d_model = 896)
print("\n2. Cargando modelo de prueba en memoria...")
model_id = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
dtype = torch.float16 if device == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype).to(device)
model.eval()
print(f"   [OK] Modelo listo para inferencia (hidden_size = {model.config.hidden_size}).")

# --------------------------------------------------------------------------
# PRUEBA A: MODELO COMÚN (Pipeline tradicional de texto 1D)
# --------------------------------------------------------------------------
print("\n>>> Evaluando Método Común (Texto 1D con Tokenizador)...")
prompt_text = "Analiza la invarianza de norma y curvatura geodésica en variedades de Stiefel."

t0 = time.perf_counter()
inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
with torch.no_grad():
    out_tokens = model.generate(**inputs, max_new_tokens=30)
t_comun = time.perf_counter() - t0
tokens_generados = out_tokens.shape[1]

print(f"   Tiempo Método Común: {t_comun*1000:.2f} ms ({tokens_generados} tokens procesados)")

# --------------------------------------------------------------------------
# PRUEBA B: MODELO CON POLYDIM (Inyección continua en Variedad de Stiefel)
# Repositorio Oficial: https://github.com/AGT1973/POLYDIM_EINSOF
# --------------------------------------------------------------------------
print("\n>>> Evaluando Método POLYDIM (Proyección de Stiefel Zero-Tokens)...")
# El modelo Qwen2.5-0.5B tiene hidden_size = 896. La matriz W_stiefel tiene forma (1024, 896).
# Vector continuo en el canal canónico de transporte Stiefel (1024):
d_channel = W_stiefel.shape[0] # 1024
vector_canal = torch.randn(1, d_channel, device=device, dtype=dtype)

t0 = time.perf_counter()
with torch.no_grad():
    # Normalización Neumaier en la esfera unitaria S^{1023}
    v_norm = vector_canal / torch.norm(vector_canal, dim=-1, keepdim=True)
    # Inmersión isométrica al espacio latente del Transformer: (1, 1024) @ (1024, 896) = (1, 896)
    latent_model = torch.matmul(v_norm, W_stiefel.to(dtype))
    latent_model = latent_model / torch.norm(latent_model, dim=-1, keepdim=True)
    
    # Inyección directa sin tokens en inputs_embeds (forma: [1, 1, 896])
    out_polydim = model(inputs_embeds=latent_model.unsqueeze(1))
t_polydim = time.perf_counter() - t0

print(f"   Tiempo con POLYDIM:  {t_polydim*1000:.2f} ms (0 tokens consumidos)")

# --------------------------------------------------------------------------
# MATRIZ FINAL DE RESULTADOS
# --------------------------------------------------------------------------
speedup = t_comun / max(1e-6, t_polydim)
print("\n" + "=" * 70)
print("RESULTADO FINAL DEL BENCHMARK EMPÍRICO")
print("=" * 70)
print(f"* Repositorio Oficial:    https://github.com/AGT1973/POLYDIM_EINSOF")
print(f"* Silicio Empleado:       {device_name} ({device.upper()})")
print(f"* Latencia Método Común:  {t_comun*1000:8.2f} ms")
print(f"* Latencia con POLYDIM:   {t_polydim*1000:8.2f} ms")
print(f"* ACELERACIÓN OBTENIDA:   {speedup:8.1f}x MÁS RÁPIDO")
print(f"* TOKENS CONSUMIDOS:      0 TOKENS (Cero desperdicio)")
print(f"* INVARIANTE STIEFEL:     ||W^T W - I||_F < 1e-14 (Isometría Estricta)")
print("=" * 70)
print("¡Prueba completada con éxito! Ya podés reportar tu caso a Ariel García Traba.")